In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from timeit import default_timer as timer

In [2]:
%pip install presidio-analyzer
%pip install presidio-anonymizer
%pip install presidio-analyzer[transformers]
%pip install docling
%pip install thefuzz
%pip install triton
%pip install torch

  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
Using cached typer-0.24.1-py3-none-any.whl (56 kB)
  Attempting uninstall: typer
    Found existing installation: typer 0.21.2
    Uninstalling typer-0.21.2:
      Successfully uninstalled typer-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docling 2.82.0 requires typer<0.22.0,>=0.12.5, but you have typer 0.24.1 which is incompatible.
  Using cached typer-0.21.2-py3-none-any.whl.metadata (16 kB)
Using cached typer-0.21.2-py3-none-any.whl (56 kB)
  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer-slim 0.24.0 requires

In [3]:
%pip install langgraph
%pip install langchain
%pip install langchain-community
%pip install langchain-google-genai
%pip install langchain-deepseek

In [24]:
from docling.document_converter import DocumentConverter

start_timer = timer()
source = "/content/Cyber Mutual Assistance NDA.pdf"

converter = DocumentConverter()
result = converter.convert(source)

context = result.document.export_to_markdown()
print(context)

[INFO] 2026-03-26 08:03:09,866 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-26 08:03:09,867 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-26 08:03:09,951 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-26 08:03:09,952 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-03-26 08:03:10,339 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-03-26 08:03:10,341 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-03-26 08:03:10,345 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-26 08:03:10,347 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-03-26 08:03:10,525 [RapidO

## Mutual Non-Disclosure and Use of Information Agreement to Support Emergency Cyber Mutual Assistance

This Non-Disclosure and Use of Information Agreement (the "Agreement'') is made and entered into as of this 15 th day of June, 2016 by and among each entity that executes and delivers the signature page to this Agreement (each, a "Participating Entity" and collectively, the "Participating Entities").

- A. Each Participating Entity is participating in a voluntary effort to assist the Electricity Subsector Coordinating Council (ESCC) in developing and implementing one or more industry initiatives to provide cyber emergency assistance to entities in the electric sector (collectively, the 'Cyber Mutual Assistance Program').
- B. In connection with the Cyber Mutual Assistance Program, each Participating Entity may voluntarily choose to request from or provide to another Participating Entity emergency cyber mutual assistance in response to a cyber emergency;
- C. The development and imple

In [25]:
from thefuzz import fuzz
from thefuzz import process
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

configuration = {
    "nlp_engine_name": "transformers",
    "models": [
        {
            "lang_code": "en",
            "model_name": {
                "spacy": "en_core_web_sm",
                "transformers": "dslim/bert-base-NER"
            }
        }
    ]
}

def find_similar_entity(entity_text, entity_map, threshold=80):
  choices = list(entity_map.keys())
  closest = process.extractOne(entity_text, choices)
  if closest and closest[1] > threshold:
    return closest[0]
  return None


def remove_overlaps(results):
  results = sorted(results, key=lambda x: (x.start, -(x.end - x.start)))
  filtered = []

  for r in results:
      if not any(not (r.end <= f.start or r.start >= f.end) for f in filtered):
          filtered.append(r)

  return filtered


def normalize_text(t):
  return t.strip().lower().replace(".", "")


entity_types = [
    "CREDIT_CARD",
    "CRYPTO",
    "EMAIL_ADDRESS",
    "IBAN_CODE",
    "IP_ADDRESS",
    "MAC_ADDRESS",
    "NRP",
    "LOCATION",
    "PERSON",
    "PHONE_NUMBER",
    "MEDICAL_LICENSE",
    "URL",
    "ORGANIZATION"
]

entity_counters = {}
entity_map = {}


provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

text = str(context)

results = analyzer.analyze(text=text, language="en")

results = remove_overlaps(results)

results = sorted(results, key=lambda x: x.start, reverse=True)


for r in results:
  if r.entity_type == "DATE_TIME" or r.score < 0.5:
      continue

  entity_text = text[r.start:r.end]
  entity_type = "UNCATEGORIZED_PII" if r.entity_type not in entity_types else r.entity_type

  similar = None
  similar = find_similar_entity(entity_text, entity_map)

  if similar:
      placeholder = entity_map[similar]
  else:
      if entity_type not in entity_counters:
          entity_counters[entity_type] = 0

      entity_counters[entity_type] += 1
      entity_map[entity_text] = f"<{entity_type}_{entity_counters[entity_type]}>"
      placeholder = entity_map[entity_text]

  text = text[:r.start] + placeholder + text[r.end:]

print(text)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


## Mutual Non-Disclosure and Use of Information Agreement to Support Emergency <ORGANIZATION_3> Assistance

This Non-Disclosure and Use of Information Agreement (the "Agreement'') is made and entered into as of this 15 th day of June, 2016 by and among each entity that executes and delivers the signature page to this Agreement (each, a "Participating Entity" and collectively, the "Participating Entities").

- A. Each Participating Entity is participating in a voluntary effort to assist the <ORGANIZATION_6> (<ORGANIZATION_5>) in developing and implementing one or more industry initiatives to provide cyber emergency assistance to entities in the electric sector (collectively, the '<ORGANIZATION_3>').
- B. In connection with the <ORGANIZATION_3>, each Participating Entity may voluntarily choose to request from or provide to another Participating <ORGANIZATION_2> emergency cyber mutual assistance in response to a cyber emergency;
- C. The development and implementation of any <ORGANIZATION

In [28]:
mapped_tuples = list(entity_map.items())
mapped_tuples

[('Information', '<ORGANIZATION_1>'),
 ('Entity', '<ORGANIZATION_2>'),
 ('U.S.C.', '<LOCATION_1>'),
 ('Cyber Mutual Assistance', '<ORGANIZATION_3>'),
 ('Entities', '<ORGANIZATION_4>'),
 ('ESCC', '<ORGANIZATION_5>'),
 ('Electricity Subsector Coordinating Council', '<ORGANIZATION_6>')]

In [27]:
from langgraph.graph import StateGraph, END, START
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage
import operator
from typing import TypedDict, Annotated, List
from pydantic import BaseModel, Field
import json

class ContractState(TypedDict):
  text: str
  comments: Annotated[List[BaseMessage], operator.add]
  legal_risk: float
  financial_risk: float
  compliance_risk: float
  final_score: float
  final_report: str


class LegalOut(BaseModel):
  comment: str = Field(
      description="Brief explanation of legal risks identified in the contract, such as unfair clauses, one-sided obligations, or lack of protections."
  )
  legal_risk: float = Field(
      description="Legal risk score between 0 and 1, where 0 means no legal risk and 1 means very high legal risk."
  )


class FinancialOut(BaseModel):
  comment: str = Field(
      description="Short explanation of financial risks including penalties, payment obligations, liability exposure, or monetary losses."
  )
  financial_risk: float = Field(
      description="Financial risk score between 0 and 1, where 0 means no financial risk and 1 means very high financial risk."
  )


class ComplianceOut(BaseModel):
  comment: str = Field(
      description="Explanation of compliance-related risks such as regulatory violations, legal obligations, or governance issues."
  )
  compliance_risk: float = Field(
      description="Compliance risk score between 0 and 1, where 0 means fully compliant and 1 means high regulatory or legal compliance risk."
  )


class Summarizer(BaseModel):
  risk_analyzed_report: str = Field(
      description="Overall summary of the contract risk combining legal, financial, and compliance aspects."
  )
  final_score: float = Field(
      description="Final aggregated risk score between 0 and 1 representing the overall risk level of the contract."
  )


from google.colab import userdata
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

llm = ChatDeepSeek(
  model="deepseek-chat",
  temperature=0,
  api_key=DEEPSEEK_API_KEY
)


def legal_agent(state: ContractState):

  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a legal risk expert."),
      ("user",
        """Analyze the legal risk (0 to 1) of the PII masked contract.

        Return:
        - legal_risk (float)
        - comment (short explanation)

        Contract:
        {contract}"""
      )
  ])

  chain = prompt | llm.with_structured_output(LegalOut)
  result = chain.invoke({"contract": state["text"]})
  return {
      "legal_risk": result.legal_risk,
      "comments": [HumanMessage(content=f"[LEGAL] {result.comment}")]
  }


def financial_agent(state: ContractState):

  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a financial risk expert."),
      ("user",
       """Analyze financial risk (payments, penalties, liabilities) of the PII masked contract.

          Return:
          - financial_risk (0 to 1)
          - comment

          Contract:
          {contract}"""
      )
  ])

  chain = prompt | llm.with_structured_output(FinancialOut)
  result = chain.invoke({"contract": state["text"]})
  return {
      "financial_risk": result.financial_risk,
      "comments": [HumanMessage(content=f"[FINANCIAL] {result.comment}")]
  }


def compliance_agent(state: ContractState):

  prompt = ChatPromptTemplate.from_messages([
      ("system", "You are a compliance expert."),
      ("user",
       """Analyze compliance risk (legal/regulatory issues) of the PII masked contract.

          Return:
          - compliance_risk (0 to 1)
          - comment

          Contract:
          {contract}"""
      )
  ])

  chain = prompt | llm.with_structured_output(ComplianceOut)
  result = chain.invoke({"contract": state["text"]})
  return {
      "compliance_risk": result.compliance_risk,
      "comments": [HumanMessage(content=f"[COMPLIANCE] {result.comment}")]
  }


def evaluator(state: ContractState):

  prompt = ChatPromptTemplate.from_messages([
      ("system", """You are a Contract Risk Evaluator Expert. You have to carefully look at risk scores provided by experts under each field.

                    COMMENTS: {comments}
                    RISK_SCORES (0 to 1 scale): {risk}

                    Now provide the final Contract Risk Evaluation Report in markdown format, with a FINAL RISK SCORE.

                    Return:
                    - final Contract Risk Evaluation Report (100-200 words)
                    - final Risk Score (0 to 1 scale)"""
      )

  ])

  chain = prompt | llm.with_structured_output(Summarizer)
  risk_scores = json.dumps({"legal_risk": state["legal_risk"], "financial_risk": state["financial_risk"], "compliance_risk": state["compliance_risk"]}, indent=2)
  result = chain.invoke({"comments": state["comments"], "risk": risk_scores})
  return {
      "final_score": result.final_score,
      "final_report": result.risk_analyzed_report
  }


graph = StateGraph(ContractState)

graph.add_node("legal", legal_agent)
graph.add_node("financial", financial_agent)
graph.add_node("compliance", compliance_agent)
graph.add_node("evaluator", evaluator)

graph.add_edge(START, "legal")
graph.add_edge("legal", "financial")
graph.add_edge("financial", "compliance")
graph.add_edge("compliance", "evaluator")
graph.add_edge("evaluator", END)

app = graph.compile()

final_state = app.invoke({
    "text": context,
    "comments": []
})

print("-"*50)
print(final_state["final_report"])
print("-"*50)
print("\n")
print(f"FINAL SCORE: {final_state["final_score"]}")
print("-"*50)

end_timer = timer()

print(f"Elapsed time: {round(end_timer-start_timer, 2)}s")

--------------------------------------------------
This Mutual Non-Disclosure Agreement for a Cyber Mutual Assistance Program presents a moderate overall risk profile with balanced protections and concerning provisions. The agreement establishes reasonable confidentiality protections for sensitive cybersecurity information exchange during emergency situations, but contains several risk factors: broad legal process exemptions with minimal 7-day notice, unilateral withdrawal rights with only 10 days notice, no warranties on information accuracy, and potential conflicts with regulatory reporting requirements. The mutual nature and specific purpose for emergency cyber assistance mitigate some risks, while the lack of direct financial obligations keeps financial exposure low. However, compliance risks are elevated due to potential conflicts with mandatory cybersecurity reporting obligations, inconsistent FOIA handling approaches, and lack of specific data classification protocols. The agree